In [ ]:
import pandas as pd
import numpy as np

# List of BED files
bed_files = [
    "/content/drive/MyDrive/peak1.bed",
    "/content/drive/MyDrive/peak2.bed",
    "/content/drive/MyDrive/peak3.bed",
    "/content/drive/MyDrive/peak4.bed",
    "/content/drive/MyDrive/peak5.bed",
    "/content/drive/MyDrive/peak6.bed",
    "/content/drive/MyDrive/peak7.bed",
    "/content/drive/MyDrive/peak8.bed"]

# Load BED files into Pandas dataframes
bed_dfs = []
for bed_file in bed_files:
    df = pd.read_csv(bed_file, sep="\t", header=None, usecols=[0, 1, 2], names=["chrom", "start", "end"])
    bed_dfs.append(df)

# Concatenate all peaks
all_peaks = pd.concat(bed_dfs)

# Sort peaks by chromosome and start position
all_peaks = all_peaks.sort_values(by=["chrom", "start", "end"]).reset_index(drop=True)

# Convert to NumPy array for efficient processing
peaks_array = all_peaks.to_numpy()

# Efficient merging of overlapping peaks
merged_peaks = []
current_chrom, current_start, current_end, count = peaks_array[0, 0], peaks_array[0, 1], peaks_array[0, 2], 1

for i in range(1, len(peaks_array)):
    chrom, start, end = peaks_array[i]

    # Check if the peak overlaps with the previous one
    if chrom == current_chrom and start <= current_end:
        current_end = max(current_end, end)
        count += 1
    else:
        merged_peaks.append([current_chrom, current_start, current_end, count])
        current_chrom, current_start, current_end, count = chrom, start, end, 1

# Add the last peak
merged_peaks.append([current_chrom, current_start, current_end, count])

# Convert merged peaks to DataFrame
merged_df = pd.DataFrame(merged_peaks, columns=["chrom", "start", "end", "count"])

# Filter peaks appearing in at least 10 samples
consensus_df = merged_df[merged_df["count"] >= 10]

# Save the consensus peaks to a new BED file
consensus_path = "consensus_peaks8.bed"
consensus_df.to_csv(consensus_path, sep="\t", index=False, header=False)
consensus_df
